# Thesis: Stored fact vs. context in GPT-2 small

## Objective

Find out how GPT-2 small resolves a conflict between a fact it has stored and a contradicting statement in the prompt, and which components carry each path.

- **Recall (R):** `The Eiffel Tower is in the city of` → the stored answer (Paris)
- **Counterfactual (C):** `The Eiffel Tower is in Rome. The Eiffel Tower is in the city of` → stored answer (Paris) or context answer (Rome)

**Behavior measure.** Logit difference at the last position, `LD = logit(stored answer) - logit(context answer)`. The **override rate** is the fraction of C prompts with `LD < 0`.

**Dataset.** 40-60 fact templates (landmark → city, country → capital, and similar) with these rules:
- Both the stored and the context answer must be a single GPT-2 token.
- Keep a template only if the stored answer is in the model's top-3 on its R prompt, so we know the model "has" the fact.
- Each template gets four prompt conditions:
  - **R:** recall, no context. e.g. "The Eiffel Tower is in the city of"
  - **C:** context contradicts the stored fact. e.g. "The Eiffel Tower is in Rome. The Eiffel Tower is in the city of"
  - **S:** context agrees with the stored fact. It is the same length as C and differs by one token. e.g. "The Eiffel Tower is in Paris. The Eiffel Tower is in the city of"
  - **N:** a neutral context sentence of the same length, to control for length and repetition. e.g. "The Eiffel Tower is very tall. The Eiffel Tower is in the city of"
  - By keeping the token lengths of C, S, and N identical, we ensure that every token sits at the exact same positional index across conditions. When we run activation patching or attention head ablations, we can cleanly separate which neural components copy the answer from the prompt (the context path) versus which components pull the answer from memory (the factual recall path), without worrying about length mismatches skewing the results.


## Hypotheses

**H1: The context overrides the stored fact in a meaningful fraction of prompts.**
- *Details:* Mean `LD` on C is lower than on R and S, and the override rate falls between about 10% and 90%. N-prompts should shift `LD` much less than C-prompts.
- *Falsified if:* the override rate is under 10% or over 90%. Then there is no competition to explain, and we adjust the prompts or switch behavior before going on.

**H2: Two separate paths. Recall runs mainly through mid-layer MLPs, and copying from context runs mainly through late-layer attention heads.**
- *Details:* Ablating the top mid-layer MLPs should lower the stored answer's logit on R but barely change the context answer's logit on C. Ablating the top late heads should do the reverse.
- *Falsified if:* the same components are the top contributors for both behaviors.

**H3: The heads that copy the context answer behave like induction heads.**
- *Details:* On C prompts these heads attend from the final token to the context answer (the token after the earlier "is in"). They should also score high on an induction test with repeated random tokens.
- *Falsified if:* the copy heads have low induction scores, or the high-scoring induction heads have no effect on the context answer's logit.

**H4: The competition is steerable in both directions.**
- *Details:* Adding a stored-answer direction to a C run raises the override-reversal rate. Adding a context-answer direction to an R run pushes toward the context answer. The effect should grow smoothly with scale.
- *Falsified if:* flips happen only at scales that wreck other predictions (check loss and top tokens), or only for a few templates.

**H5: Recall and copy correspond to distinct SAE features at layer 8.**
- *Details:* Some features fire on the stored-answer position in R but not in N, and others fire on C prompts at the context answer. Ablating the first group should lower stored-answer `LD`, and ablating the second should do the opposite.
- *Falsified if:* the candidate features fire equally in N, or ablating them has no more effect than ablating random features.

**H6: A small circuit explains most of the behavior.**
- *Details:* Roughly 10-20% of heads plus a few MLPs restore most of the clean `LD` when everything else is mean-ablated.
- *Falsified if:* restoring takes most of the network, or the circuit works on the templates used to find it but not on held-out templates.

## Sequence of steps

Every session starts with a torch warm-up on hand-checkable numbers, then a short concept cell with predictions, then experiments with inline code explanations and a control.

| Step | What we do | What it achieves |
|---|---|---|
| **0. Build the behavior** | Build the dataset and the four conditions, filter by top-3 knowledge, compute `LD` and override rate. | Tests H1 and fixes the baseline numbers. Gives a go / no-go for the thesis. |
| **1. Ablation and attribution** | Ablate each head and MLP and measure the change in `LD` on R and on C. Add gradient attribution as a cross-check. | A first ranking of components per path. First test of H2. |
| **2. Residual stream, logit lens, direct logit attribution** | Decode every layer's residual stream, find where the context answer overtakes the stored one, and credit components for the final `LD`. | Locates when the competition resolves and which components write each answer. |
| **3. Neuron analysis** | Take the top MLP neurons from steps 4-5 and read their top-activating contexts, with random weak activations as a fairness check. | Says what the recall path computes. Warns about false readings from best examples only. |
| **4. Activation patching** | Patch activations between C and S (same length, one token different) component by component, and across layers and positions. | The causal version of step 4. Confirms H2 or narrows it. |
| **5. Attention analysis, induction heads** | Compute induction scores on repeated random tokens, then inspect attention of the top copy heads on C prompts. | Tests H3, with the random-token test as a control. |
| **6. Steering vectors** | Build stored-answer and context-answer directions, add them at several scales, and measure flip rate and side effects. | Tests H4 and shows where steering breaks. |
| **7. Path patching** | Patch edges (which head feeds which) along the copy path and the recall path. | Turns a list of important components into a wiring diagram. |
| **8. Reverse-engineering** | Combine the above into a candidate circuit, mean-ablate everything else, and test on held-out templates. | Tests H6, including whether the circuit generalizes. |
| **9. SAE features and transcoders** | Use the pretrained layer 8 SAE (steps 2-7 of your SAE doc): select features by R-vs-N and C-vs-S differences, read their contexts and decoder effects, then ablate them. | Tests H5 and ties the circuit to readable features. |
| **Write-up** | A short report: behavior, one table per hypothesis with its outcome, negative results, limits. | The thesis, with failed predictions kept in. |

**Limits to state in the write-up:** one small model, template prompts, two answers per prompt, and effects measured on `LD` only. Patching and ablation show causal effects on this metric, not a full account of how the model works.

In [ ]:
!pip install transformer_lens

In [2]:
import torch
from transformer_lens import TransformerBridge

device = "cuda" if torch.cuda.is_available() else "cpu"
model = TransformerBridge.boot_transformers("gpt2", device = device)
TransformerBridge.enable_compatibility_mode(model)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [3]:
import torch

# (landmark, stored city). Each city must be ONE GPT-2 token, leading space included.
data_set = [
    ("The Eiffel Tower"," Paris"),("The Louvre"," Paris"),("The Arc de Triomphe"," Paris"),("The Moulin Rouge"," Paris"),("The Musee d'Orsay"," Paris"),
    ("Big Ben"," London"),("Tower Bridge"," London"),("Buckingham Palace"," London"),("Westminster Abbey"," London"),("The British Museum"," London"),("Trafalgar Square"," London"),
    ("The Colosseum"," Rome"),("The Trevi Fountain"," Rome"),("The Spanish Steps"," Rome"),("Piazza Navona"," Rome"),("Castel Sant'Angelo"," Rome"),
    ("The Brandenburg Gate"," Berlin"),("The Reichstag"," Berlin"),("Checkpoint Charlie"," Berlin"),("Alexanderplatz"," Berlin"),("Museum Island"," Berlin"),
    ("The Acropolis"," Athens"),("The Parthenon"," Athens"),("The Panathenaic Stadium"," Athens"),
    ("The Kremlin"," Moscow"),("Red Square"," Moscow"),("Saint Basil's Cathedral"," Moscow"),("The Bolshoi Theatre"," Moscow"),("Gorky Park"," Moscow"),
    ("The Prado Museum"," Madrid"),("Puerta del Sol"," Madrid"),("Retiro Park"," Madrid"),("The Reina Sofia Museum"," Madrid"),
    ("Schonbrunn Palace"," Vienna"),("Hofburg Palace"," Vienna"),("Belvedere Palace"," Vienna"),("The Prater"," Vienna"),
    ("Wenceslas Square"," Prague"),("The Charles Bridge"," Prague"),
    ("The Guinness Storehouse"," Dublin"),("Temple Bar"," Dublin"),("Phoenix Park"," Dublin"),("Kilmainham Gaol"," Dublin"),("Croke Park"," Dublin"),
    ("Holyrood Palace"," Edinburgh"),("Arthur's Seat"," Edinburgh"),("The Royal Mile"," Edinburgh"),
    ("The Rialto Bridge"," Venice"),("St. Mark's Basilica"," Venice"),("The Doge's Palace"," Venice"),("The Bridge of Sighs"," Venice"),
    ("The Uffizi Gallery"," Florence"),("The Ponte Vecchio"," Florence"),("Palazzo Vecchio"," Florence"),
    ("The Blue Mosque"," Istanbul"),("Topkapi Palace"," Istanbul"),("The Grand Bazaar"," Istanbul"),("The Galata Tower"," Istanbul"),("Hagia Sophia"," Istanbul"),
    ("The Egyptian Museum"," Cairo"),("Al-Azhar Mosque"," Cairo"),("Tahrir Square"," Cairo"),
    ("Shibuya Crossing"," Tokyo"),("Senso-ji Temple"," Tokyo"),("The Meiji Shrine"," Tokyo"),("Ueno Zoo"," Tokyo"),
    ("Tiananmen Square"," Beijing"),("The Temple of Heaven"," Beijing"),("The Summer Palace"," Beijing"),("The Forbidden City"," Beijing"),
    ("The Bund"," Shanghai"),("The Oriental Pearl Tower"," Shanghai"),("Yu Garden"," Shanghai"),
    ("Bondi Beach"," Sydney"),("Taronga Zoo"," Sydney"),("Darling Harbour"," Sydney"),("The Sydney Opera House"," Sydney"),
    ("The CN Tower"," Toronto"),("Rogers Centre"," Toronto"),("Casa Loma"," Toronto"),
    ("Willis Tower"," Chicago"),("Wrigley Field"," Chicago"),("Millennium Park"," Chicago"),("Navy Pier"," Chicago"),("Soldier Field"," Chicago"),
    ("Fenway Park"," Boston"),("Faneuil Hall"," Boston"),("The Paul Revere House"," Boston"),
    ("The Space Needle"," Seattle"),("Pike Place Market"," Seattle"),
    ("The Lincoln Memorial"," Washington"),("The Jefferson Memorial"," Washington"),
    ("The Liberty Bell"," Philadelphia"),("Independence Hall"," Philadelphia"),
    ("The Burj Khalifa"," Dubai"),("The Burj Al Arab"," Dubai"),("Palm Jumeirah"," Dubai"),
    ("The Western Wall"," Jerusalem"),("The Dome of the Rock"," Jerusalem"),("The Mount of Olives"," Jerusalem"),
    ("Marina Bay Sands"," Singapore"),("The Merlion"," Singapore"),("Gardens by the Bay"," Singapore"),
    ("The Grand Palace"," Bangkok"),("Wat Arun"," Bangkok"),
    ("The Red Fort"," Delhi"),("India Gate"," Delhi"),("The Qutub Minar"," Delhi"),
    ("Gyeongbokgung Palace"," Seoul"),("Changdeokgung Palace"," Seoul"),
    ("The Gateway of India"," Mumbai"),
    ("The Atomium"," Brussels"),("Manneken Pis"," Brussels"),
    ("Belem Tower"," Lisbon"),("Jeronimos Monastery"," Lisbon"),
    ("Fisherman's Bastion"," Budapest"),("The Chain Bridge"," Budapest"),("Szechenyi Baths"," Budapest"),
    ("Marienplatz"," Munich"),("The Hofbrauhaus"," Munich"),("Nymphenburg Palace"," Munich"),
    ("Akershus Fortress"," Oslo"),("The Vasa Museum"," Stockholm"),("Tivoli Gardens"," Copenhagen"),("Nyhavn"," Copenhagen"),
    ("The Rijksmuseum"," Amsterdam"),("The Van Gogh Museum"," Amsterdam"),
    ("The Johnson Space Center"," Houston"),("The Georgia Aquarium"," Atlanta"),
]

# raises if the city is not exactly one token
def city_id(c):
  try:
    return model.to_single_token(c)
  except Exception:
    return None

# scores for the next token: [batch 0, last position]
def last_scores(x):
  with torch.no_grad():
    return (model(x)[0, -1])

# LD = stored - context; positive = stored wins
# ranks: 0 = top-1
def measure(prompt, stored, context):
  last = last_scores(prompt)
  s, c = city_id(stored), city_id(context)
  ld = (last[s] - last[c]).item()
  return ld, (last > last[s]).sum().item(), (last > last[c]).sum().item()

# token count, no BOS
def ntok(s):
    return len(model.to_tokens(s, prepend_bos=False)[0])   # [0] picks the row of tokens; len counts them

# True if the answer appears inside the landmark name
def leaks(subj, city):
    return city.strip().lower() in subj.lower()



## 0.3 Build and freeze the dataset

**Purpose.** Produce the dataset used by every later step. Nothing about H1 is tested here.

**Rules.**
1. Candidates: the 129 (landmark, city) pairs. Drop pairs where the city is not a single GPT-2 token, the city name appears in the landmark name (answer leak), or the city is " Washington" (ambiguous).
2. Knowledge filter on R (`<landmark> is in the city of`): strict tier = stored city in the top 3 (rank 0-2); relaxed tier = rank 3-4. Strict fills first.
3. Cap of 3 templates per stored city, in listed order, so no city dominates.
4. Context city: a different kept city, chosen to balance usage across cities (fixed seed), never one that appears in the landmark name.
5. N: a *false* statement of the context city about another landmark, with a different true city from this template's, as close in token length as possible.
6. Validity asserts: C and S have equal token length; all four prompts end with the same words.
7. Save `dataset.json` with token ids, R rank and tier per template.

**Frozen after this cell.** No template is added or removed based on how later experiments turn out; otherwise results would be biased by selection.

**Limits.** Facts GPT-2 small knows well, concentrated in famous cities; relaxed-tier facts are known only weakly.


In [4]:
import collections, random
# --- Step 1: knowledge filter on R (stored city must be in the top 3) ---
print("# --- Step 1: knowledge filter on R (stored city must be in the top 3) ---")
EXCLUDE = {" Washington"}                      # ambiguous city name: also a state and a surname
kept, skipped = [], []

for sub, city in data_set:
  cid = city_id(city)
  if cid is None:
    skipped.append((sub, f"- city {city} is not a single token"))
    continue
  if city in EXCLUDE:
        skipped.append((sub, f"{city.strip()} excluded as ambiguous")); continue
  if leaks(sub, city):
        skipped.append((sub, f"city {city} name appears in the landmark name")); continue
  last = last_scores(f"{sub} is in the city of")
  rank = (last > last[cid]).sum().item()
  kept.append(dict(subj=sub, stored=city, rank=rank))
  #we will skip later based upon strict and relazed ranking
#   if rank <= 2: #keep else discard
#     kept.append(dict(subj=sub, stored=city, rank=rank))
#   else:
#     skipped.append((sub, f"-{city} has rank {rank}"))
# print(f"kept - {len(kept)} of {len(data_set)} and skipped -{len(skipped)}")

# rank <= 2 means top-3; rank <= 4 means top-5
STRICT_MAX, RELAXED_MAX, PER_CITY = 2, 4, 3

# ---- 2. Choose templates: strict tier first, then relaxed, at most PER_CITY per city ----
chosen, count = [], collections.Counter()
for lo, high, tier in ((0, STRICT_MAX, "strict"), (STRICT_MAX+1,RELAXED_MAX, "relaxed" )):
  for data in kept:
    if lo <= data["rank"] <= high and count[data["stored"]] < PER_CITY:
      chosen.append(dict(data, tier=tier))
      count[data["stored"]] += 1
print("candidates:", len(data_set), "| valid:", len(kept), "| dropped as invalid:", len(skipped), "| chosen:", len(chosen))

# ---- 3. Build the four conditions for each template ----
#sort by city
import random, json, collections

# Rebuild from the filtered templates; drop everything the earlier pass added (including the broken N)
base = [dict(subj=e["subj"], stored=e["stored"], rank=e["rank"], tier=e["tier"]) for e in chosen]


pool = sorted(set(e["stored"] for e in base))              # every city that has at least one kept template
for e in base:                                             # loud check: no landmark name contains any pool city
    for c in pool:
        assert not leaks(e["subj"], c), f"{e['subj']} contains the pool city {c}"

rng = random.Random(0)                    # same seed and candidate order as before -> same context cities
rng_n = random.Random(1)                  # separate generator for N, so N changes do not alter the context cities
usage = collections.Counter()             # how often each city has served as the context city
n_usage = collections.Counter()           # how often each landmark has served as the N distractor

for i, e in enumerate(base):
    subj, stored = e["subj"], e["stored"]

    # context city: any pool city except this template's own; prefer the least-used so far
    cands = [c for c in pool if c != stored]
    least = min(usage[c] for c in cands)
    ctx = rng.choice([c for c in cands if usage[c] == least])
    usage[ctx] += 1

    # N distractor: a different landmark whose true city is neither the context city nor this template's city,
    # so "<other> is in the city of <ctx>" is a false statement
    ok = [j for j, o in enumerate(base) if j != i and o["stored"] not in (ctx, stored)]
    gaps = {j: abs(ntok(base[j]["subj"]) - ntok(subj)) for j in ok}      # token-length gap to this landmark
    close = [j for j in ok if gaps[j] == min(gaps.values())]             # all landmarks tied for the smallest gap
    least_n = min(n_usage[j] for j in close)                             # among those, prefer the least-used
    j = rng_n.choice([k for k in close if n_usage[k] == least_n])
    n_usage[j] += 1
    other = base[j]["subj"]

    tail = f"{subj} is in the city of"                                   # shared ending
    P = {"R": tail,
         "C": f"{subj} is in the city of{ctx}. {tail}",                  # context contradicts the stored fact
         "S": f"{subj} is in the city of{stored}. {tail}",               # context agrees (one token differs from C)
         "N": f"{other} is in the city of{ctx}. {tail}"}                 # false statement about another landmark
    assert ntok(P["C"]) == ntok(P["S"]), "C and S must have equal length"
    assert all(p.endswith(" is in the city of") for p in P.values()), "all prompts must share the ending"
    dlen = ntok(P["N"]) - ntok(P["C"])                                   # N minus C, in tokens
    assert dlen == ntok(other) - ntok(subj), "dlen should equal the distractor's length minus the landmark's"
    e.update(id=i, ctx=ctx, other=other, prompts=P, stored_id=city_id(stored), ctx_id=city_id(ctx), dlen=dlen)

# ---- report ----
dl = collections.Counter(e["dlen"] for e in base)
print(f"{len(base)} templates over {len(pool)} cities | strict {sum(e['tier'] == 'strict' for e in base)} "
      f"| relaxed {sum(e['tier'] == 'relaxed' for e in base)}")
print("N minus C length (tokens):", dict(sorted(dl.items())), "| exact matches:", f"{dl[0]}/{len(base)}")
print("N distractor usage:", dict(n_usage.most_common()))
print("context city usage:", dict(sorted((c.strip(), n) for c, n in usage.items())))
print(f"\n{'id':>3} {'tier':8}{'landmark':26}{'toks':>5}  {'N uses':26}{'toks':>5}{'dlen':>5}")
for e in base:
    print(f"{e['id']:3d} {e['tier']:8}{e['subj']:26}{ntok(e['subj']):5d}  {e['other']:26}{ntok(e['other']):5d}{e['dlen']:+5d}")

chosen = base                                                            # later cells use `chosen`


# --- Step 1: knowledge filter on R (stored city must be in the top 3) ---
candidates: 129 | valid: 126 | dropped as invalid: 3 | chosen: 40
40 templates over 24 cities | strict 32 | relaxed 8
N minus C length (tokens): {-2: 1, -1: 1, 0: 38} | exact matches: 38/40
N distractor usage: {38: 2, 2: 2, 3: 2, 7: 1, 19: 1, 32: 1, 10: 1, 17: 1, 29: 1, 39: 1, 24: 1, 37: 1, 16: 1, 1: 1, 4: 1, 33: 1, 21: 1, 9: 1, 20: 1, 30: 1, 15: 1, 5: 1, 11: 1, 18: 1, 13: 1, 0: 1, 6: 1, 12: 1, 8: 1, 31: 1, 14: 1, 26: 1, 36: 1, 23: 1, 34: 1, 22: 1, 25: 1}
context city usage: {'Amsterdam': 1, 'Athens': 1, 'Atlanta': 1, 'Berlin': 2, 'Boston': 2, 'Cairo': 1, 'Chicago': 1, 'Delhi': 2, 'Dubai': 1, 'Dublin': 2, 'Houston': 2, 'London': 2, 'Moscow': 2, 'Mumbai': 2, 'Munich': 2, 'Paris': 2, 'Prague': 2, 'Rome': 2, 'Seattle': 2, 'Seoul': 1, 'Shanghai': 2, 'Sydney': 1, 'Tokyo': 2, 'Toronto': 2}

 id tier    landmark                   toks  N uses                     toks dlen
  0 strict  The Eiffel Tower              5  Pi

In [5]:
with open("dataset_frozen.json", "w") as f:
    json.dump([{k: e[k] for k in ("id", "tier", "rank", "subj", "stored", "ctx", "other", "stored_id", "ctx_id", "dlen", "prompts")}
               for e in base], f, indent=1)
print("\nsaved dataset_frozen.json with", len(base), "templates")


saved dataset_frozen.json with 40 templates


## 0.4 Test H1 on the frozen dataset

**Purpose.** Measure how often GPT-2 small follows a statement in the prompt over a fact it already learned during training, using only `dataset_template_10_1.json` (40 templates × 4 prompt types = 160 prompts).

For each prompt, we compute the **Logit Difference** ($\text{LD} = \text{logit}(\text{stored city}) - \text{logit}(\text{context city})$) at the final token position:
* **Positive LD ($\text{LD} > 0$):** The model favors its stored memory over the city given in the prompt.
* **Negative LD ($\text{LD} < 0$):** The context overrides the model's memory, pushing the prompt's city ahead.

We then report the following five measurements:

1. **Average preference across prompt types (Mean LD for R, C, S, N):**
   * Report the average LD for each of the four prompt conditions—**R** (Recall baseline), **C** (Conflicting context), **S** (Supportive context), and **N** (Neutral mention)—along with a **95% bootstrap confidence interval**.
   * *What the 95% confidence interval means here:* Because 40 templates are only a finite sample of all possible facts we could test, the sample average carries some sampling noise. A **95% confidence interval** gives the plausible range of values for the true underlying average across similar templates; if we repeated this experiment on many fresh 40-template datasets, 95% of those intervals would capture the true average. We build this range via **bootstrap resampling** (drawing 10,000 simulated 40-template datasets with replacement) so we do not have to assume the logit differences follow a neat bell curve.

2. **How often the context wins (Override rate):**
   * Report the **override rate**—the percentage of templates where $\text{LD} < 0$ under the conflicting prompt (**C**)—alongside a **95% Wilson score interval**, plus the share of templates where the context city becomes the model's #1 overall predicted token.
   * *Why we target a 95% Wilson interval:* Just as with the mean, we want a **95% confidence interval** showing the plausible range for the true override rate. Standard confidence interval formulas break down on smaller sample sizes ($n = 32, 38,$ or $40$) and can produce impossible numbers below 0% or above 100% when rates are near the extremes. The **Wilson score interval** ($\frac{\hat{p} + \frac{z^2}{2n} \pm z\sqrt{\frac{\hat{p}(1-\hat{p})}{n} + \frac{z^2}{4n^2}}}{1 + \frac{z^2}{n}}$, with $z = 1.96$ for 95% confidence) stays strictly between 0% and 100% and reliably achieves its 95% target coverage even on small template counts.

3. **Template-by-template shifts and statistical significance ($\text{C}-\text{R}$, $\text{S}-\text{R}$, $\text{N}-\text{R}$):**
   * Because some cities are naturally more common in English than others, raw LD scores vary from template to template. By subtracting each template's own baseline (**R**) from its other conditions, baseline city popularity cancels out and we isolate the exact effect of the added context:
     * **$\text{C}-\text{R}$ (Conflict shift — total pull of a false claim):** Measures how strongly a false statement pushes the model away from its stored fact toward the context city. A **negative value** means the conflicting context successfully weakens the stored fact, boosts the false city, or both.
     * **$\text{S}-\text{R}$ (Supportive shift — boost from confirmation):** Measures whether confirming the true fact in the prompt makes the model even more confident than bare recall. A **positive value** shows that the model actively combines in-context confirmation with its stored memory, whereas a value near **zero** means the model was already at peak confidence.
     * **$\text{N}-\text{R}$ (Priming shift — effect of mere word repetition):** Measures how much simply mentioning the distractor city in an unrelated sentence nudges the model away from its stored fact. A **negative value** captures shallow word-association or recency bias rather than genuine factual persuasion.
   * *How we test whether these shifts are real (95% CIs & p-values):* For each shift, we report the average change, its **95% bootstrap confidence interval** (if the entire 95% range stays away from 0, the shift is reliably non-zero), and **two-sided p-values**. A **p-value** tells us the probability of seeing a shift at least this large purely by random chance if the prompt manipulation actually had zero effect ($H_0: \Delta = 0$). We compute p-values using two distribution-free checks—a **paired permutation test** (10,000 random sign-flips of each template's shift) and a **Wilcoxon signed-rank test** (which protects against a single extreme template skewing the average)—and apply the **Holm–Bonferroni adjustment** so testing three shifts at once does not inflate our false-positive rate above 5%.

4. **How much of the override is just word priming (Priming share):**
   * Compute $\text{mean}(\text{N}-\text{R}) / \text{mean}(\text{C}-\text{R})$ along with a **95% bootstrap confidence interval** (recalculating the ratio inside each of the 10,000 bootstrap samples).
   * *What this tells us:* It expresses what percentage of the total conflict drop ($\text{C}-\text{R}$) is already explained just by mentioning the rival city name in the prompt ($\text{N}-\text{R}$). We also run a one-sided paired test ($H_1: \text{mean}(\text{C}-\text{R}) < \text{mean}(\text{N}-\text{R})$) to verify that an actual false claim shifts the model significantly more than merely mentioning the city name.

5. **Checking sensitivity across data subsets (Subset robustness):**
   * Re-run all metrics, 95% intervals, and p-values across three slices of the dataset to ensure our conclusions do not hinge on borderline templates:
     * **All templates** ($n = 40$)
     * **Strict tier only** ($n = 32$, excluding lower-confidence templates)
     * **Length-matched templates** ($n = 38$, excluding the two templates whose prompt lengths differ across conditions)

**Rule.** H1 is supported if the override rate on **C** falls between **10% and 90%** *and* its entire **95% Wilson confidence interval** also stays strictly inside **10% to 90%**. H1 is falsified if the observed override rate falls below 10% or above 90% (and treated as inconclusive if the rate itself is inside 10–90% but one edge of its 95% interval crosses 10% or 90%). Whatever happens, the result stays in the write-up as is. Any change to the prompts becomes a separate, clearly labeled exploratory run (`0.5`), never a replacement for this result.


In [14]:
import json, numpy as np, torch
from scipy import stats

with open("dataset_frozen.json") as f:
    data = json.load(f)
res = []

with torch.no_grad():
  for d in data:
    sid, cid = d["stored_id"], d["ctx_id"]
    row = {}
    for k in "RCNS":
      last = model(d["prompts"][k])[0, -1]
      row[k] = dict(ld=(last[sid] - last[cid]).item(),               # LD = stored - context; positive = stored wins
                          rank_s=(last > last[sid]).sum().item(),          # 0 = the stored city is the top prediction
                          rank_c=(last > last[cid]).sum().item(),          # 0 = the context city is the top prediction
                          top1=model.tokenizer.decode(int(last.argmax())))
    res.append(row)
json.dump(res, open("results_frozen.json", "w"), indent=1)
print("measured", len(res), "templates x 4 prompts =", 4 * len(res))



measured 40 templates x 4 prompts = 160


In [16]:
def boot_ci(stat, *arrays, B=5000, seed=0):
    # Bootstrap CI: resample whole templates with replacement, recompute `stat` each time.
    # paired=True keeps the arrays aligned (the same template in R, C, S and N moves together).
    # vectorized=True means `stat` must accept an `axis` argument, which lets SciPy run all resamples at once.
    kw = dict(n_resamples=B, confidence_level=0.95, method="percentile", paired=True, vectorized=True)
    try:
        r = stats.bootstrap(arrays, stat, rng=np.random.default_rng(seed), **kw)    # newer SciPy
    except TypeError:
        r = stats.bootstrap(arrays, stat, random_state=seed, **kw)                   # older SciPy used this name
    return r.confidence_interval.low, r.confidence_interval.high
mean = lambda a, axis=-1: a.mean(axis=axis)

# ===== Report on one subset of templates =====
def report(keep, label):
    ix = [i for i, d in enumerate(data) if keep(d)]
    n = len(ix)
    LD = {k: np.array([res[i][k]["ld"] for i in ix]) for k in "RCSN"}      # plain NumPy arrays for SciPy
    print(f"\n=== {label}: n = {n} templates ===")
    for k in "RCSN":
        lo, hi = boot_ci(mean, LD[k])
        print(f"{k}: mean LD {LD[k].mean():+6.2f}  (95% CI {lo:+.2f} to {hi:+.2f})  | LD<0 on {(LD[k] < 0).sum()}/{n}")
    for a, b in (("C", "R"), ("S", "R"), ("N", "R"), ("C", "N")):
        dd = LD[a] - LD[b]                                                  # paired: same template, so differences are meaningful
        lo, hi = boot_ci(mean, dd)
        pw = stats.wilcoxon(dd).pvalue                                      # signed-rank test: is the typical shift nonzero?
        print(f"paired shift {a}-{b}: {dd.mean():+6.2f}  (95% CI {lo:+.2f} to {hi:+.2f}) | Wilcoxon p = {pw:.2g}")
    over = int((LD["C"] < 0).sum()); rate = over / n
    ci = stats.binomtest(over, n).proportion_ci(confidence_level=0.95, method="wilson")
    p_hi = stats.binomtest(over, n, p=0.9, alternative="greater").pvalue    # evidence the true rate exceeds 90%
    p_lo = stats.binomtest(over, n, p=0.1, alternative="less").pvalue       # evidence the true rate is below 10%
    top1 = sum(res[i]["C"]["rank_c"] == 0 for i in ix)
    print(f"override rate (C: LD<0): {over}/{n} = {rate:.0%}  (95% Wilson {ci.low:.0%} to {ci.high:.0%})")
    print(f"  test rate > 90%: p = {p_hi:.3g} | test rate < 10%: p = {p_lo:.3g}")
    print(f"context city is the top-1 prediction on C: {top1}/{n}")
    share = lambda r, c, nn, axis=-1: (nn.mean(axis=axis) - r.mean(axis=axis)) / (c.mean(axis=axis) - r.mean(axis=axis))
    slo, shi = boot_ci(share, LD["R"], LD["C"], LD["N"])
    sh = (LD["N"].mean() - LD["R"].mean()) / (LD["C"].mean() - LD["R"].mean())
    print(f"priming share mean(N-R)/mean(C-R): {sh:.0%}  (95% bootstrap {slo:.0%} to {shi:.0%})")
    if 0.10 <= ci.low and ci.high <= 0.90:
        print("H1 verdict: SUPPORTED (rate and its whole interval lie within 10-90%)")
    elif 0.10 <= rate <= 0.90:
        print("H1 verdict: INCONCLUSIVE (rate inside 10-90% but the interval crosses a threshold)")
    else:
        print("H1 verdict: OUTSIDE the pre-registered range -> falsified as worded; record it, do not edit the dataset")

In [18]:
report(lambda d: True, "all 40")


=== all 40: n = 40 templates ===
R: mean LD  +4.88  (95% CI +4.18 to +5.62)  | LD<0 on 0/40
C: mean LD  -9.06  (95% CI -9.99 to -8.15)  | LD<0 on 40/40
S: mean LD +11.93  (95% CI +11.06 to +12.81)  | LD<0 on 0/40
N: mean LD  -3.02  (95% CI -3.66 to -2.38)  | LD<0 on 37/40
paired shift C-R: -13.94  (95% CI -14.94 to -12.94) | Wilcoxon p = 1.8e-12
paired shift S-R:  +7.05  (95% CI +6.15 to +7.94) | Wilcoxon p = 1.8e-12
paired shift N-R:  -7.90  (95% CI -8.80 to -7.05) | Wilcoxon p = 1.8e-12
paired shift C-N:  -6.04  (95% CI -6.75 to -5.34) | Wilcoxon p = 1.8e-12
override rate (C: LD<0): 40/40 = 100%  (95% Wilson 91% to 100%)
  test rate > 90%: p = 0.0148 | test rate < 10%: p = 1
context city is the top-1 prediction on C: 40/40
priming share mean(N-R)/mean(C-R): 57%  (95% bootstrap 52% to 61%)
H1 verdict: OUTSIDE the pre-registered range -> falsified as worded; record it, do not edit the dataset


In [20]:
report(lambda d: d["tier"] == "strict", "strict tier only")


=== strict tier only: n = 32 templates ===
R: mean LD  +4.77  (95% CI +3.96 to +5.61)  | LD<0 on 0/32
C: mean LD  -9.01  (95% CI -10.02 to -7.96)  | LD<0 on 32/32
S: mean LD +11.85  (95% CI +10.85 to +12.92)  | LD<0 on 0/32
N: mean LD  -3.00  (95% CI -3.71 to -2.36)  | LD<0 on 30/32
paired shift C-R: -13.79  (95% CI -14.96 to -12.59) | Wilcoxon p = 4.7e-10
paired shift S-R:  +7.08  (95% CI +6.07 to +8.11) | Wilcoxon p = 4.7e-10
paired shift N-R:  -7.78  (95% CI -8.80 to -6.70) | Wilcoxon p = 4.7e-10
paired shift C-N:  -6.01  (95% CI -6.79 to -5.23) | Wilcoxon p = 4.7e-10
override rate (C: LD<0): 32/32 = 100%  (95% Wilson 89% to 100%)
  test rate > 90%: p = 0.0343 | test rate < 10%: p = 1
context city is the top-1 prediction on C: 32/32
priming share mean(N-R)/mean(C-R): 56%  (95% bootstrap 51% to 61%)
H1 verdict: OUTSIDE the pre-registered range -> falsified as worded; record it, do not edit the dataset


### Summary of GPT-2 Small Behavior (`0.4` Frozen Dataset Results)

Across all 40 templates (and identically within the 32-template strict tier), **GPT-2 small exhibits complete context dominance rather than a mixed tug-of-war between memory and context**, falsifying the pre-registered H1 range of 10–90%:

* **Reliable Baseline Recall (`R`):** Without conflicting context, GPT-2 small consistently retrieves its stored factual knowledge, achieving a mean Logit Difference (LD) of **+4.88** (95% CI: +4.18 to +5.62; strict tier: **+4.77**) with **0/40** baseline errors ($\text{LD} < 0$).
* **Total Override Under Conflict (`C`):** Presenting a false context city overrides the model's parametric memory **100% of the time** (**40/40**, 95% Wilson CI: 91–100%; strict tier: **32/32**, 95% Wilson CI: 89–100%). Mean LD swings by **-13.94 logits** down to **-9.06** ($p = 1.8 \times 10^{-12}$), and the false context city becomes the model's **#1 overall predicted token** on every single template (40/40).
* **Substantial Headroom for Confirmation (`S`):** Confirming the true city in context boosts confidence far above bare recall, raising mean LD by **+7.05 logits** up to **+11.93** ($p = 1.8 \times 10^{-12}$; strict tier: **+7.08**). Bare parametric recall is not saturated; corroborating context strongly reinforces stored facts.
* **Two-Part Override Mechanism (Priming vs. Semantic Conflict):** Decomposing the **-13.94 logit** conflict shift reveals two distinct effects:
  * **Surface Token Priming (`N - R`):** Simply mentioning the distractor city in a non-conflicting frame drops mean LD by **-7.90 logits** down to **-3.02** ($p = 1.8 \times 10^{-12}$), flipping $\text{LD} < 0$ on **37/40** templates (92.5%). Shallow lexical priming alone accounts for **57%** (95% bootstrap CI: 52–61%; strict tier: **56%**) of the total conflict shift.
  * **Genuine Semantic Contradiction (`C - N`):** Framing the distractor city as a factual claim adds a further **-6.04 logit** penalty ($p = 1.8 \times 10^{-12}$; strict tier: **-6.01**), accounting for the remaining **43%** of the shift and pushing every remaining template into full override.
* **Subset Invariance:** Restricting the analysis to the **32 strict-tier templates** changes mean logit shifts by less than **0.15 logits** and priming share by **1 percentage point** (57% vs. 56%), confirming that GPT-2 small's extreme susceptibility to context is a systematic model behavior rather than an artifact of borderline prompts.
